# Pipelines - Revamped Version


ce notebook sert pour tester le backend des pipelines

# Initialisation

In [1]:
%pip install tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## Imports Librairies

In [2]:
import os
import sys
from tqdm import tqdm
import glob
import numpy as np
from PIL import Image
from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

import importlib
import inspect


"""import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)
"""



'import warnings\nfrom sklearn.exceptions import ConvergenceWarning\nwarnings.filterwarnings("ignore", category=ConvergenceWarning)\n'

## Ajouter le répertoire racine du projet au chemin Python ( à remplacer par toml)

In [3]:
# src est au niveau parent du répertoire notebooks
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

## Imports Features et Transformateurs

In [4]:
# Imports avec chemins absolus (pas de . ou .. dans les notebooks)
from src.features.Pipelines.loading_pipelines import *
from src.features.Pipelines.Transformateurs import *
from src.features.Pipelines.Visualisations import *

## Imports Visualisations

In [5]:
from src.features.Pipelines.Visualisations.Visu_Pipeline_Seed_Support import *
from src.features.Pipelines.Visualisations.Visu_Conf_Matrix import *

## Checkup Répertoire de Travail

In [6]:
print(f"\n📁 Chemin du projet ajouté: {project_root}")
print(f"📂 Répertoire de travail actuel: {os.getcwd()}")


📁 Chemin du projet ajouté: c:\Users\Léna\Documents\A_Repos\DS_COVID
📂 Répertoire de travail actuel: c:\Users\Léna\Documents\A_Repos\DS_COVID\notebooks


## Checkup Transformateurs (Optionnel)

In [7]:
from src.features.Pipelines.Visualisations.Visu_Pipeline_Seed_Support import *

print_transformers(project_root)

🔍 Découverte automatique des transformateurs:

📁 Module: image_augmentation
------------------------------
  ✅ ImageAugmenter 🎲 (seed via constructor)
  ✅ ImageRandomCropper 🎲 (seed via constructor)

📁 Module: image_features
------------------------------
  ✅ ImageHistogram
  ✅ ImagePCA 🎲 (seed via constructor)
  ✅ ImageStandardScaler
  ✅ PCA 🎲 (seed via constructor)
  ✅ StandardScaler

📁 Module: image_loaders
------------------------------
  ✅ ImageLoader

📁 Module: image_preprocessing
------------------------------
  ✅ AddChannelDim
  ✅ ImageBinarizer
  ✅ ImageFlattener
  ✅ ImageMasker
  ✅ ImageNormalizer
  ✅ ImageResizer

📁 Module: keras_classifiers
------------------------------
  ✅ LabelEncoder

📁 Module: tensorflow_data_augmenter
------------------------------
  ✅ TensorFlowDataAugmenter 🎲 (seed via constructor)

📁 Module: tensorflow_feature_extractor
------------------------------
  ✅ TensorFlowFeatureExtractor 🎲 (seed via constructor)

📁 Module: utilities
----------------------

## Configs

### Variables Environnement

In [8]:
root_dir = '../data/raw/COVID-19_Radiography_Dataset/COVID-19_Radiography_Dataset/'

In [9]:
seed = 42

# Main

## Chargement Paths

In [10]:
X, masks, y = load_paths_data_raw(root_dir)

In [11]:
print(f"Nombre d'images: {len(X)}")
print(f"Labels uniques: {set(y)}")

Nombre d'images: 21165
Labels uniques: {'lung_opacity', 'covid', 'normal', 'viral pneumonia'}


## Split des jeux de données

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

Train size: 16932, Test size: 4233


## Chargement des pipelines disponibles

In [13]:
# Découverte automatique des pipelines disponibles
registry_config = discover_available_pipelines(os.path.join(project_root, "src/features/Pipelines/Configs_Pipelines/"))
display_pipeline_structure(registry_config)

Pipelines disponibles:

Catégorie: BASIC

  simple: Pipeline simple pour classification rapide 
     Fichier: pipeline_simple.json


Catégorie: COMPOSITE

  composite_example: Pipeline composite utilisant d'autres pipelines comme sous-étapes 
     Fichier: pipeline_composite_example.json

  composite_no_viz: Pipeline composite sans visualisation (plus rapide pour les tests) 
     Fichier: pipeline_composite_no_viz.json


Catégorie: DATA_AUGMENTATION

  augmented: Pipeline avec augmentation de données et masquage 
     Fichier: pipeline_augmented.json


Catégorie: DEEP_LEARNING

  keras_full_pipeline: Pipeline complet utilisant TensorFlow pour l'augmentation et la classification 
     Fichier: keras_full_pipeline.json

  projet_covid: Pipeline optimal pour la détection COVID sur radios pulmonaires avec masques 
     Fichier: pipeline_projet_covid.json

  tensorflow: Pipeline utilisant des transformateurs TensorFlow pour l'extraction de caractéristiques et la classification 
     Fichier

## Séléction Pipeline

In [14]:
# Chargement du pipeline par défaut avec seed
default_pipeline_name = "projet_covid"
# default_pipeline_name = "tensorflow"

default_config = load_pipeline_config(registry_config[default_pipeline_name]["file"])
pipeline = create_pipeline_from_config(default_config, masks)

print(f"\nPipeline par défaut chargé:\n  {default_config['name']}")
print(f"\nDescription:\n  {default_config['description']}")
print(f"\nNombre d'étapes:\n  {len(pipeline.steps)}")

for step_name, step in pipeline.steps:
    print(f"  - Étape: {step_name} | Type: {type(step).__name__}")

# Appliquer la seed
set_pipeline_random_state(pipeline, seed)

TypeError: TensorFlowFeatureExtractor.__init__() got an unexpected keyword argument 'weights'

In [ ]:
pipeline

## Training Pipeline

In [ ]:
# Entraînement du pipeline
pipeline.fit(X_train, y_train)

## Prédiction Pipeline

In [ ]:
# Prédiction
y_pred = pipeline.predict(X_test)

## Matrice De Confusion

In [ ]:
# Matrice de confusion visuelle
plot_confusion_matrix(y_test, y_pred)


## Quelques exemples d'images (Vrai vs. Prédit)

In [ ]:
# Afficher quelques images test avec prédiction et vrai label
nbr_images = 3
for i in range(nbr_images):
    img = Image.open(X_test[i])
    plt.imshow(img, cmap='gray')
    plt.title(f"Vrai: {y_test[i]} / Prédit: {y_pred[i]}")
    plt.axis('off')
    plt.show()